# simulate_rw_coef

> 对应代码：`EconometricsCode/code_in_notes/Chap.29/simulate_rw_coef.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.29`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.29")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先清空内存，并新建一个名为 betas 的 frame，用来存放每次模拟得到的 6 个统计量（3 种样本量下的 √T(ρ̂−1) 与 T(ρ̂−1)），这样就不必在主数据集里反复 preserve/restore。

In [ ]:
%%stata
clear all
frame create betas b1 b2 b3 v1 v2 v3

设置 5000 个观测、固定随机种子，并生成时间变量 t 后声明为时间序列；误差项 e 先初始化为缺失值，留待在循环中反复填充。

In [ ]:
%%stata
set obs 5000
set seed 15634145
gen t = _n
tsset t
gen e = .

我们进行 1000 次蒙特卡洛模拟。每次模拟先用 rnormal() 生成标准正态误差，再按随机游走 y_t = y_{t−1} + e_t 递推生成序列（初始值为 0）。随后分别用前 500、前 1500 和全部 5000 个观测对 y 关于 L.y 做不含常数项的回归：单位根过程的一阶自回归系数真值为 1，而 OLS 估计量以超一致（super-consistent）的速率收敛，因此 √T(ρ̂−1) 会随 T 增大而向 0 收缩，T(ρ̂−1) 则收敛到一个非退化的极限分布（与 Dickey-Fuller 分布相关）。我们把两种标准化后的统计量存入局部宏并 post 到 betas 框架中，最后删掉 y 以便下次重新生成。整个循环用 qui{} 抑制中间输出。

In [ ]:
%%stata
forvalues i=1/1000{
	qui{
		replace e = rnormal()
		gen y = 0 if _n==1
		replace y = L.y + e if _n > 1
		reg y L.y if _n<=500, noc
		local b1 = sqrt(500)*(_b[L.y]-1)
		local v1 = 500*(_b[L.y]-1)
		reg y L.y if _n<=1500, noc
		local b2 = sqrt(1500)*(_b[L.y]-1)
		local v2 = 1500*(_b[L.y]-1)
		reg y L.y, noc
		local b3 = sqrt(5000)*(_b[L.y]-1)
		local v3 = 5000*(_b[L.y]-1)
		frame post betas (`b1') (`b2') (`b3') (`v1') (`v2') (`v3')
		drop y
	}
}

切换到 betas 框架，用核密度图比较三种样本量下 √T(ρ̂−1) 的分布。可以看到随着 T 增大，密度越来越集中在 0 附近，直观地展示了单位根情形下 √T 标准化并不是正确的收敛速率。

In [ ]:
%%stata
frame change betas
// twoway (hist b1, color(gs3)) (hist b2, color(gs7%60)) (hist b3, color(gs12%60)), legend(label(1 "T=500") label(2 "T=1500") label(3 "T=5000")) xline(1)
twoway (kdensity b1, lp(solid) lc(gs0)) (kdensity b2,lp(dash) lc(gs0)) (kdensity b3, lp("-..-") lc(gs0)), legend(label(1 "T=500") label(2 "T=1500") label(3 "T=5000")) xline(0)

再画出 T(ρ̂−1) 的核密度。三条曲线几乎重合，说明 T 倍的偏差具有稳定的非退化极限分布——这正是单位根过程超一致性的表现，其极限分布向左偏斜，反映了 ρ̂ 向下的偏误。

In [ ]:
%%stata
twoway (kdensity v1, lp(solid) lc(gs0)) (kdensity v2,lp(dash) lc(gs0)) (kdensity v3, lp("-..-") lc(gs0)), legend(label(1 "T=500") label(2 "T=1500") label(3 "T=5000")) xline(0)